In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    """Locate the FAME repository root from the current working directory."""
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "results").exists()
        ):
            return candidate
    raise RuntimeError(
        "FAME repository root not found. Run this notebook from inside a clone "
        "of the FAME repository."
    )

REPO_ROOT = find_repo_root()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
REPRODUCED_DIR = REPO_ROOT / "reproduced"
REPRODUCED_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = REPRODUCED_DIR / "energy" / "temporal_replication"

required = {
    "predictive": RESULT_DIR / "predictive_metrics_native_scale.csv",
    "robustness": RESULT_DIR / "robustness_margin_voll_summary.csv",
    "boundary": RESULT_DIR / "boundary_selection_summary.csv",
    "freezes": RESULT_DIR / "theta_freezes_margin_voll.csv",
    "test": RESULT_DIR / "test_decisions_margin_voll.csv",
    "cal": RESULT_DIR / "calibration_surface_margin_voll.csv",
}

missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Reproduced Energy outputs are missing. Run "
        "code/energy/FAME_energy_temporal_replication.ipynb first.\n"
        + "\n".join(missing)
    )

predictive = pd.read_csv(required["predictive"])
robustness = pd.read_csv(required["robustness"])
boundary = pd.read_csv(required["boundary"])
freezes = pd.read_csv(required["freezes"])
test = pd.read_csv(required["test"])
cal = pd.read_csv(required["cal"])

print("Reproduced Energy outputs loaded from:", RESULT_DIR)


In [ ]:
pred_wide = predictive.pivot(
    index=["replication_id","model"],
    columns="stage",
    values=["rmse","mae","mape","bias"]
).reset_index()

pred_wide.columns = [
    "_".join(str(x) for x in col if str(x)!="")
    if isinstance(col, tuple) else col
    for col in pred_wide.columns
]

pred_wide["delta_rmse"] = (
    pred_wide["rmse_test"] - pred_wide["rmse_calibration"]
)

pred_wide["relative_rmse_change_pct"] = (
    pred_wide["delta_rmse"] /
    pred_wide["rmse_calibration"] * 100
)

display(pred_wide.sort_values(["model","replication_id"]))

pred_summary = (
    pred_wide.groupby("model",as_index=False)
    .agg(
        mean_rmse_cal=("rmse_calibration","mean"),
        mean_rmse_test=("rmse_test","mean"),
        mean_relative_rmse_change_pct=("relative_rmse_change_pct","mean"),
        median_relative_rmse_change_pct=("relative_rmse_change_pct","median"),
        sd_relative_rmse_change_pct=("relative_rmse_change_pct","std"),
    )
)

display(pred_summary)


In [ ]:
theta_by_rep = (
    freezes.groupby(["replication_id","model"],as_index=False)
    .agg(
        mean_theta=("theta_doc","mean"),
        median_theta=("theta_doc","median"),
        sd_theta_operational=("theta_doc","std"),
        min_theta=("theta_doc","min"),
        max_theta=("theta_doc","max"),
    )
)

theta_summary = (
    theta_by_rep.groupby("model",as_index=False)
    .agg(
        mean_theta=("mean_theta","mean"),
        sd_theta_between_replications=("mean_theta","std"),
        mean_within_replication_sd=("sd_theta_operational","mean"),
        global_min_theta=("min_theta","min"),
        global_max_theta=("max_theta","max"),
    )
)

display(theta_by_rep)
display(theta_summary)


In [ ]:
test_summary = (
    test.groupby(
        ["replication_id","margin","voll_multiplier","model","strategy"],
        as_index=False
    )
    .agg(mean_loss=("realized_loss","mean"))
)

test_wide = test_summary.pivot(
    index=["replication_id","margin","voll_multiplier","model"],
    columns="strategy",
    values="mean_loss"
).reset_index()

test_wide["gain_pct"] = (
    (test_wide["baseline"] - test_wide["FAME-DOC"])
    / test_wide["baseline"] * 100
)

test_wide["transferred"] = test_wide["gain_pct"] > 0

global_operational = (
    test_wide.groupby("model",as_index=False)
    .agg(
        n_cases=("transferred","count"),
        transfer_rate=("transferred","mean"),
        mean_gain_pct=("gain_pct","mean"),
        median_gain_pct=("gain_pct","median"),
        min_gain_pct=("gain_pct","min"),
        max_gain_pct=("gain_pct","max"),
    )
)

global_operational["transfer_rate_pct"] = (
    100 * global_operational["transfer_rate"]
)

display(global_operational)


In [ ]:
rep_gain = (
    test_wide.groupby(["replication_id","model"],as_index=False)
    .agg(
        mean_test_gain_pct=("gain_pct","mean"),
        median_test_gain_pct=("gain_pct","median"),
        min_test_gain_pct=("gain_pct","min"),
        transfer_rate_within_rep=("transferred","mean"),
    )
)

rep_gain["all_operational_conditions_positive"] = (
    rep_gain["transfer_rate_within_rep"] == 1.0
)

display(rep_gain.sort_values(["model","replication_id"]))

rep_level_summary = (
    rep_gain.groupby("model",as_index=False)
    .agg(
        n_temporal_replications=("replication_id","count"),
        temporal_replications_all_positive=(
            "all_operational_conditions_positive","sum"
        ),
        mean_replication_gain_pct=("mean_test_gain_pct","mean"),
        sd_replication_gain_pct=("mean_test_gain_pct","std"),
        worst_replication_gain_pct=("mean_test_gain_pct","min"),
    )
)

display(rep_level_summary)


In [ ]:
master = (
    pred_wide[
        [
            "replication_id","model",
            "rmse_calibration","rmse_test",
            "relative_rmse_change_pct"
        ]
    ]
    .merge(
        theta_by_rep[
            [
                "replication_id","model",
                "mean_theta","sd_theta_operational"
            ]
        ],
        on=["replication_id","model"],
        validate="one_to_one"
    )
    .merge(
        rep_gain[
            [
                "replication_id","model",
                "mean_test_gain_pct",
                "min_test_gain_pct",
                "transfer_rate_within_rep",
                "all_operational_conditions_positive"
            ]
        ],
        on=["replication_id","model"],
        validate="one_to_one"
    )
)

display(master.sort_values(["model","replication_id"]))

OUT = RESULT_DIR / "scientific_synthesis"
OUT.mkdir(exist_ok=True)

master.to_csv(
    OUT/"table_predictive_representation_operational_transfer.csv",
    index=False
)


In [ ]:
plt.figure(figsize=(8,5))

for model in master["model"].unique():
    s = master[master["model"]==model]
    plt.scatter(
        s["relative_rmse_change_pct"],
        s["mean_test_gain_pct"],
        label=model
    )

plt.axhline(0,linestyle="--",linewidth=1)
plt.axvline(0,linestyle=":",linewidth=1)
plt.xlabel("Relative RMSE change: Calibration → Test (%)")
plt.ylabel("Mean FAME operational gain in Test (%)")
plt.title("Predictive transferability vs decision-representation transferability")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(
    OUT/"fig_predictive_vs_operational_transfer.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
plot_df = rep_gain.copy()
plot_df["test_year"] = (
    plot_df["replication_id"]
    .str.replace("E","",regex=False)
    .astype(int)
)

for model in plot_df["model"].unique():
    s = plot_df[plot_df["model"]==model].sort_values("test_year")

    plt.figure(figsize=(8,4.5))
    plt.axhline(0,linewidth=1)
    plt.plot(
        s["test_year"],
        s["mean_test_gain_pct"],
        marker="o"
    )
    plt.xlabel("Test year")
    plt.ylabel("Mean FAME gain across operational scenarios (%)")
    plt.title(f"Temporal replication of FAME transfer — {model}")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(
        OUT/f"fig_temporal_transfer_{model.lower()}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


In [ ]:
theta_plot = theta_by_rep.copy()
theta_plot["test_year"] = (
    theta_plot["replication_id"]
    .str.replace("E","",regex=False)
    .astype(int)
)

for model in theta_plot["model"].unique():
    s = theta_plot[theta_plot["model"]==model].sort_values("test_year")

    plt.figure(figsize=(8,4.5))
    plt.axhline(0,linestyle="--",linewidth=1)
    plt.plot(
        s["test_year"],
        s["mean_theta"],
        marker="o"
    )
    plt.xlabel("Test year")
    plt.ylabel(r"Mean calibrated $\widehat\theta$")
    plt.title(f"Temporal trajectory of decision representation — {model}")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(
        OUT/f"fig_theta_temporal_{model.lower()}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


In [ ]:
boundary_global = (
    boundary.groupby("model",as_index=False)
    .agg(
        max_boundary_rate_pct=("boundary_rate_pct","max"),
        mean_boundary_rate_pct=("boundary_rate_pct","mean"),
        max_theta=("max_theta","max"),
        min_theta=("min_theta","min"),
    )
)

display(boundary_global)

if (boundary_global["max_boundary_rate_pct"] > 0).any():
    print("ATENÇÃO: ainda há seleção de fronteira.")
else:
    print("Boundary audit: OK — nenhuma seleção na fronteira.")
